# Zava VoC Semantic Retrieval Audit

This notebook evaluates whether Zava's semantic retrieval system can reliably identify cross-language customer feedback themes across reviews and support conversations.

It includes:
- Cross-language embedding and retrieval over mixed sources
- Top-k qualitative inspection
- Hand-labeled Precision@5 evaluation
- Per-language and per-theme breakdowns

## 1) Setup

Install dependencies if needed, then import packages.

In [38]:
# Uncomment if needed in a fresh environment
%pip install -q pandas numpy sentence-transformers scikit-learn tqdm matplotlib

from __future__ import annotations

import os
import re
from pathlib import Path
from typing import List, Dict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 220)

Note: you may need to restart the kernel to use updated packages.


## 2) Configuration

Set your input data path and multilingual embedding model.

In [39]:
DATA_PATH = Path('data/voc_corpus.csv')
MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
TOP_K = 5

# Optional: define business themes with multilingual query prompts
THEME_QUERIES = [
    {'query_id': 'delivery_delay', 'theme': 'Delivery delays', 'query_text': 'My order arrived late and delivery tracking was poor', 'query_language': 'en'},
    {'query_id': 'refund_friction', 'theme': 'Refund friction', 'query_text': 'No pude obtener mi reembolso rapidamente', 'query_language': 'es'},
    {'query_id': 'support_quality', 'theme': 'Support quality', 'query_text': 'Le service client etait lent et peu utile', 'query_language': 'fr'},
    {'query_id': 'app_usability', 'theme': 'App usability', 'query_text': 'Die App ist schwer zu benutzen', 'query_language': 'de'},
    {'query_id': 'billing_error', 'theme': 'Billing issues', 'query_text': 'Mi hanno addebitato due volte', 'query_language': 'it'},
]

OUTPUT_DIR = Path('artifacts/semantic_retrieval_audit')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RETRIEVAL_RESULTS_PATH = OUTPUT_DIR / 'retrieval_topk_results.csv'
LABEL_TEMPLATE_PATH = OUTPUT_DIR / 'precision_at_5_labels_template.csv'
FINAL_LABELS_PATH = OUTPUT_DIR / 'precision_at_5_labels.csv'

## 3) Load VoC Corpus

Expected columns in `voc_corpus.csv`:
- `doc_id`: unique record id
- `source`: `review` or `support_conversation`
- `language`: language code (en, es, fr, de, it, etc.)
- `text`: customer feedback text
- `theme_label` (optional): known theme label for weak validation

If no file exists, a small multilingual demo corpus is generated so the notebook can run end-to-end.

In [40]:
def clean_text(s: str) -> str:
    s = str(s or '').strip()
    s = re.sub(r'\s+', ' ', s)
    return s

if DATA_PATH.exists():
    voc_df = pd.read_csv(DATA_PATH)
else:
    demo_rows = [
        ('r1', 'review', 'en', 'Delivery was late by a week and no clear tracking updates', 'delivery_delay'),
        ('r2', 'review', 'es', 'El pedido llego tarde y nadie respondio en soporte', 'delivery_delay'),
        ('r3', 'review', 'fr', 'Le remboursement a pris trop de temps', 'refund_friction'),
        ('r4', 'review', 'de', 'Die App ist verwirrend und der Checkout funktioniert nicht', 'app_usability'),
        ('r5', 'review', 'it', 'Ho ricevuto un doppio addebito sulla carta', 'billing_error'),
        ('s1', 'support_conversation', 'en', 'Agent could not resolve my refund issue after multiple chats', 'refund_friction'),
        ('s2', 'support_conversation', 'es', 'El servicio al cliente no ayudo y fue muy lento', 'support_quality'),
        ('s3', 'support_conversation', 'fr', 'Le service client etait poli mais inefficace', 'support_quality'),
        ('s4', 'support_conversation', 'de', 'Lieferung verspatet und Tracking war ungenau', 'delivery_delay'),
        ('s5', 'support_conversation', 'it', 'L assistenza non risponde e il rimborso non arriva', 'refund_friction'),
        ('r6', 'review', 'en', 'Excellent packaging and fast shipping', 'other_positive'),
        ('s6', 'support_conversation', 'en', 'Issue fixed quickly, thanks for the help', 'other_positive'),
    ]
    voc_df = pd.DataFrame(demo_rows, columns=['doc_id', 'source', 'language', 'text', 'theme_label'])

required_cols = {'doc_id', 'source', 'language', 'text'}
missing = required_cols - set(voc_df.columns)
if missing:
    raise ValueError(f'Missing required columns: {missing}')

voc_df = voc_df.copy()
voc_df['doc_id'] = voc_df['doc_id'].astype(str)
voc_df['source'] = voc_df['source'].astype(str).str.lower().str.strip()
voc_df['language'] = voc_df['language'].astype(str).str.lower().str.strip()
voc_df['text'] = voc_df['text'].map(clean_text)
voc_df = voc_df[voc_df['text'].str.len() > 0].drop_duplicates(subset=['doc_id']).reset_index(drop=True)

print(f'Loaded {len(voc_df)} VoC documents')
voc_df.head(10)

Loaded 12 VoC documents


,doc_id,source,language,text,theme_label
0,r1,review,en,Delivery was late by a week and no clear tracking updates,delivery_delay
1,r2,review,es,El pedido llego tarde y nadie respondio en soporte,delivery_delay
2,r3,review,fr,Le remboursement a pris trop de temps,refund_friction
3,r4,review,de,Die App ist verwirrend und der Checkout funktioniert nicht,app_usability
4,r5,review,it,Ho ricevuto un doppio addebito sulla carta,billing_error
5,s1,support_conversation,en,Agent could not resolve my refund issue after multiple chats,refund_friction
6,s2,support_conversation,es,El servicio al cliente no ayudo y fue muy lento,support_quality
7,s3,support_conversation,fr,Le service client etait poli mais inefficace,support_quality
8,s4,support_conversation,de,Lieferung verspatet und Tracking war ungenau,delivery_delay
9,s5,support_conversation,it,L assistenza non risponde e il rimborso non arriva,refund_friction


## 4) Build Embeddings and Retrieval Index

Use a multilingual sentence transformer so semantically similar text can match across languages.

In [41]:
model = SentenceTransformer(MODEL_NAME)

doc_texts = voc_df['text'].tolist()
doc_embeddings = model.encode(
    doc_texts,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

doc_embeddings.shape

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.48s/it]


(12, 384)

## 5) Run Semantic Retrieval For Theme Queries

Retrieves top-5 documents for each theme query and exports a review file.

In [42]:
queries_df = pd.DataFrame(THEME_QUERIES)
query_embeddings = model.encode(
    queries_df['query_text'].tolist(),
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

results = []
for i, q in queries_df.iterrows():
    sims = cosine_similarity(query_embeddings[i].reshape(1, -1), doc_embeddings).flatten()
    top_idx = np.argsort(-sims)[:TOP_K]

    for rank, idx in enumerate(top_idx, start=1):
        row = voc_df.iloc[idx]
        results.append({
            'query_id': q['query_id'],
            'theme': q['theme'],
            'query_text': q['query_text'],
            'query_language': q['query_language'],
            'rank': rank,
            'doc_id': row['doc_id'],
            'doc_source': row['source'],
            'doc_language': row['language'],
            'doc_text': row['text'],
            'similarity': float(sims[idx]),
            'auto_theme_label': row['theme_label'] if 'theme_label' in row.index else None,
        })

results_df = pd.DataFrame(results).sort_values(['query_id', 'rank']).reset_index(drop=True)
results_df.to_csv(RETRIEVAL_RESULTS_PATH, index=False)

print(f'Wrote retrieval results: {RETRIEVAL_RESULTS_PATH}')
results_df.head(15)

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.86it/s]

Wrote retrieval results: artifacts/semantic_retrieval_audit/retrieval_topk_results.csv


,query_id,theme,query_text,query_language,rank,doc_id,doc_source,doc_language,doc_text,similarity,auto_theme_label
0,app_usability,App usability,Die App ist schwer zu benutzen,de,1,r4,review,de,Die App ist verwirrend und der Checkout funktioniert nicht,0.616049,app_usability
1,app_usability,App usability,Die App ist schwer zu benutzen,de,2,s3,support_conversation,fr,Le service client etait poli mais inefficace,0.418035,support_quality
2,app_usability,App usability,Die App ist schwer zu benutzen,de,3,s2,support_conversation,es,El servicio al cliente no ayudo y fue muy lento,0.398423,support_quality
3,app_usability,App usability,Die App ist schwer zu benutzen,de,4,s6,support_conversation,en,"Issue fixed quickly, thanks for the help",0.239807,other_positive
4,app_usability,App usability,Die App ist schwer zu benutzen,de,5,s4,support_conversation,de,Lieferung verspatet und Tracking war ungenau,0.211218,delivery_delay
5,billing_error,Billing issues,Mi hanno addebitato due volte,it,1,r5,review,it,Ho ricevuto un doppio addebito sulla carta,0.649726,billing_error
6,billing_error,Billing issues,Mi hanno addebitato due volte,it,2,s1,support_conversation,en,Agent could not resolve my refund issue after multiple chats,0.295329,refund_friction
7,billing_error,Billing issues,Mi hanno addebitato due volte,it,3,r3,review,fr,Le remboursement a pris trop de temps,0.216576,refund_friction
8,billing_error,Billing issues,Mi hanno addebitato due volte,it,4,s4,support_conversation,de,Lieferung verspatet und Tracking war ungenau,0.160612,delivery_delay
9,billing_error,Billing issues,Mi hanno addebitato due volte,it,5,s3,support_conversation,fr,Le service client etait poli mais inefficace,0.107950,support_quality


## 6) Build Hand-Label Template (Precision@5)

Human annotators should mark `is_relevant` as 1 or 0 for each retrieved result.

Guideline:
- 1: document clearly matches the query theme intent
- 0: does not match, or only weakly related

In [43]:
label_template = results_df[[
    'query_id', 'theme', 'query_text', 'query_language',
    'rank', 'doc_id', 'doc_source', 'doc_language', 'doc_text', 'similarity'
]].copy()
label_template['is_relevant'] = np.nan
label_template['annotator'] = ''
label_template['notes'] = ''

label_template.to_csv(LABEL_TEMPLATE_PATH, index=False)
print(f'Wrote label template: {LABEL_TEMPLATE_PATH}')
label_template.head(10)

Wrote label template: artifacts/semantic_retrieval_audit/precision_at_5_labels_template.csv


,query_id,theme,query_text,query_language,rank,doc_id,doc_source,doc_language,doc_text,similarity,is_relevant,annotator,notes
0,app_usability,App usability,Die App ist schwer zu benutzen,de,1,r4,review,de,Die App ist verwirrend und der Checkout funktioniert nicht,0.616049,NaN,,
1,app_usability,App usability,Die App ist schwer zu benutzen,de,2,s3,support_conversation,fr,Le service client etait poli mais inefficace,0.418035,NaN,,
2,app_usability,App usability,Die App ist schwer zu benutzen,de,3,s2,support_conversation,es,El servicio al cliente no ayudo y fue muy lento,0.398423,NaN,,
3,app_usability,App usability,Die App ist schwer zu benutzen,de,4,s6,support_conversation,en,"Issue fixed quickly, thanks for the help",0.239807,NaN,,
4,app_usability,App usability,Die App ist schwer zu benutzen,de,5,s4,support_conversation,de,Lieferung verspatet und Tracking war ungenau,0.211218,NaN,,
5,billing_error,Billing issues,Mi hanno addebitato due volte,it,1,r5,review,it,Ho ricevuto un doppio addebito sulla carta,0.649726,NaN,,
6,billing_error,Billing issues,Mi hanno addebitato due volte,it,2,s1,support_conversation,en,Agent could not resolve my refund issue after multiple chats,0.295329,NaN,,
7,billing_error,Billing issues,Mi hanno addebitato due volte,it,3,r3,review,fr,Le remboursement a pris trop de temps,0.216576,NaN,,
8,billing_error,Billing issues,Mi hanno addebitato due volte,it,4,s4,support_conversation,de,Lieferung verspatet und Tracking war ungenau,0.160612,NaN,,
9,billing_error,Billing issues,Mi hanno addebitato due volte,it,5,s3,support_conversation,fr,Le service client etait poli mais inefficace,0.107950,NaN,,


## 6b) Optional: Auto-Fill Baseline Labels

Use this to generate a starter `precision_at_5_labels.csv` automatically.

Labeling rule:
- `is_relevant = 1` when `query_id == auto_theme_label`
- `is_relevant = 0` otherwise

This is a baseline for speed, not a substitute for human annotation quality checks.

In [44]:
template = pd.read_csv(LABEL_TEMPLATE_PATH)
retrieval = pd.read_csv(RETRIEVAL_RESULTS_PATH)

keys = ['query_id', 'rank', 'doc_id']
merged = template.merge(
    retrieval[keys + ['auto_theme_label']],
    on=keys,
    how='left',
    validate='one_to_one'
)

merged['is_relevant'] = (merged['query_id'] == merged['auto_theme_label']).astype(int)
merged['annotator'] = 'auto-baseline-theme-match'
merged['notes'] = 'Auto-filled baseline; review manually before final reporting.'
merged = merged.drop(columns=['auto_theme_label'])

merged.to_csv(FINAL_LABELS_PATH, index=False)

p5_baseline = merged.groupby('query_id', as_index=False)['is_relevant'].mean()
macro_p5_baseline = float(p5_baseline['is_relevant'].mean())

print(f'Wrote completed labels file: {FINAL_LABELS_PATH}')
print(f'Macro Precision@{TOP_K} baseline: {macro_p5_baseline:.3f}')
display(p5_baseline.rename(columns={'is_relevant': 'precision_at_5'}).sort_values('precision_at_5', ascending=False))

Wrote completed labels file: artifacts/semantic_retrieval_audit/precision_at_5_labels.csv
Macro Precision@5 baseline: 0.360


,query_id,precision_at_5
2,delivery_delay,0.6
4,support_quality,0.4
3,refund_friction,0.4
0,app_usability,0.2
1,billing_error,0.2


## 7) Evaluate Precision@5 From Hand Labels

After annotation, save the completed file to `FINAL_LABELS_PATH` and run this cell.

Precision@5 for a query: 
$$
P@5 = \frac{\sum_{i=1}^{5} rel_i}{5}
$$
where $rel_i \in \{0,1\}$.

In [45]:
if not FINAL_LABELS_PATH.exists():
    print('No completed label file found yet.')
    print(f'1) Open template: {LABEL_TEMPLATE_PATH}')
    print(f'2) Fill is_relevant with 0/1 and save as: {FINAL_LABELS_PATH}')
else:
    labeled = pd.read_csv(FINAL_LABELS_PATH)

    required = {'query_id', 'rank', 'is_relevant', 'doc_language', 'theme'}
    missing = required - set(labeled.columns)
    if missing:
        raise ValueError(f'Missing columns in labeled file: {missing}')

    labeled = labeled.copy()
    labeled = labeled[labeled['rank'] <= TOP_K]

    # Strict binary coercion for annotation quality control
    labeled['is_relevant'] = pd.to_numeric(labeled['is_relevant'], errors='coerce')
    invalid = labeled[~labeled['is_relevant'].isin([0, 1])]
    if len(invalid) > 0:
        display(invalid.head(10))
        raise ValueError('Found invalid is_relevant values. Use only 0 or 1.')

    p5_by_query = (
        labeled.groupby('query_id', as_index=False)['is_relevant']
        .mean()
        .rename(columns={'is_relevant': 'precision_at_5'})
        .sort_values('precision_at_5', ascending=False)
    )

    macro_p5 = p5_by_query['precision_at_5'].mean()

    p5_by_theme = (
        labeled.groupby(['theme', 'query_id'], as_index=False)['is_relevant']
        .mean()
        .groupby('theme', as_index=False)['is_relevant']
        .mean()
        .rename(columns={'is_relevant': 'avg_precision_at_5'})
        .sort_values('avg_precision_at_5', ascending=False)
    )

    p5_by_doc_language = (
        labeled.groupby('doc_language', as_index=False)['is_relevant']
        .mean()
        .rename(columns={'is_relevant': 'avg_precision_at_5'})
        .sort_values('avg_precision_at_5', ascending=False)
    )

    print(f'Macro Precision@{TOP_K}: {macro_p5:.3f}')
    print('\nPrecision@5 by query')
    display(p5_by_query)

    print('\nAverage Precision@5 by theme')
    display(p5_by_theme)

    print('\nAverage Precision@5 by retrieved document language')
    display(p5_by_doc_language)

Macro Precision@5: 0.360

Precision@5 by query


,query_id,precision_at_5
2,delivery_delay,0.6
4,support_quality,0.4
3,refund_friction,0.4
0,app_usability,0.2
1,billing_error,0.2



Average Precision@5 by theme


,theme,avg_precision_at_5
2,Delivery delays,0.6
4,Support quality,0.4
3,Refund friction,0.4
0,App usability,0.2
1,Billing issues,0.2



Average Precision@5 by retrieved document language


,doc_language,avg_precision_at_5
4,it,0.500000
0,de,0.400000
3,fr,0.400000
2,es,0.285714
1,en,0.250000


## 8) Optional: Error Analysis Slice

Surface likely false positives to inspect drift by source/language.

In [46]:
if FINAL_LABELS_PATH.exists():
    labeled = pd.read_csv(FINAL_LABELS_PATH)
    labeled['is_relevant'] = pd.to_numeric(labeled['is_relevant'], errors='coerce')

    false_positives = labeled[(labeled['is_relevant'] == 0)].sort_values('similarity', ascending=False)
    print(f'Potential false positives: {len(false_positives)}')
    display(false_positives.head(20))
else:
    print('Run this section after completing annotations.')

Potential false positives: 16


,query_id,theme,query_text,query_language,rank,doc_id,doc_source,doc_language,doc_text,similarity,is_relevant,annotator,notes
13,delivery_delay,Delivery delays,My order arrived late and delivery tracking was poor,en,4,s2,support_conversation,es,El servicio al cliente no ayudo y fue muy lento,0.566977,0,auto-baseline-theme-match,Auto-filled baseline; review manually before final reporting.
22,support_quality,Support quality,Le service client etait lent et peu utile,fr,3,s5,support_conversation,it,L assistenza non risponde e il rimborso non arriva,0.510593,0,auto-baseline-theme-match,Auto-filled baseline; review manually before final reporting.
17,refund_friction,Refund friction,No pude obtener mi reembolso rapidamente,es,3,r5,review,it,Ho ricevuto un doppio addebito sulla carta,0.504197,0,auto-baseline-theme-match,Auto-filled baseline; review manually before final reporting.
18,refund_friction,Refund friction,No pude obtener mi reembolso rapidamente,es,4,r2,review,es,El pedido llego tarde y nadie respondio en soporte,0.489728,0,auto-baseline-theme-match,Auto-filled baseline; review manually before final reporting.
19,refund_friction,Refund friction,No pude obtener mi reembolso rapidamente,es,5,s2,support_conversation,es,El servicio al cliente no ayudo y fue muy lento,0.484675,0,auto-baseline-theme-match,Auto-filled baseline; review manually before final reporting.
14,delivery_delay,Delivery delays,My order arrived late and delivery tracking was poor,en,5,r6,review,en,Excellent packaging and fast shipping,0.458589,0,auto-baseline-theme-match,Auto-filled baseline; review manually before final reporting.
23,support_quality,Support quality,Le service client etait lent et peu utile,fr,4,s4,support_conversation,de,Lieferung verspatet und Tracking war ungenau,0.432314,0,auto-baseline-theme-match,Auto-filled baseline; review manually before final reporting.
24,support_quality,Support quality,Le service client etait lent et peu utile,fr,5,r2,review,es,El pedido llego tarde y nadie respondio en soporte,0.424751,0,auto-baseline-theme-match,Auto-filled baseline; review manually before final reporting.
1,app_usability,App usability,Die App ist schwer zu benutzen,de,2,s3,support_conversation,fr,Le service client etait poli mais inefficace,0.418035,0,auto-baseline-theme-match,Auto-filled baseline; review manually before final reporting.
2,app_usability,App usability,Die App ist schwer zu benutzen,de,3,s2,support_conversation,es,El servicio al cliente no ayudo y fue muy lento,0.398423,0,auto-baseline-theme-match,Auto-filled baseline; review manually before final reporting.


## 8b) SQL MCP Cross-Check Calculations

This section incorporates two SQL MCP tools into the audit evidence:
- `find_similar_docs_by_doc_id` (direct MCP tool)
- `execute_entity` for `FindSimilarDocsByDocId` (general SQL MCP execute path)

Because notebook Python cannot invoke MCP tools directly, the values below are captured from MCP tool executions and used for reproducible calculations in-notebook.

In [ ]:
# Captured MCP outputs (TopN=5) for seed DocIds 1, 2, 3
# Tool A: mcp_sql_mcp_serve_find_similar_docs_by_doc_id
mcp_find_top5 = {
    1: [1, 24, 11, 15, 9],
    2: [2, 7, 18, 21, 3],
    3: [3, 7, 2, 21, 6],
}

mcp_find_distances = {
    1: [0.0, 0.16656166315078735, 0.24704509973526, 0.32534313201904297, 0.3423503637313843],
    2: [0.0, 0.23766088485717773, 0.3196555972099304, 0.3268164396286011, 0.3391536474227905],
    3: [0.0, 0.22551000118255615, 0.3391536474227905, 0.3419368863105774, 0.3731940984725952],
}

# Tool B: mcp_sql_mcp_serve_execute_entity(entity='FindSimilarDocsByDocId', ...)
mcp_execute_top5_doc2 = [2, 7, 18, 21, 3]

# 1) Mean non-self distance by seed
rows = []
for seed_doc, dists in mcp_find_distances.items():
    non_self = [d for d in dists if d > 0]
    rows.append({
        'seed_doc_id': seed_doc,
        'mean_nonself_distance_at_5': float(np.mean(non_self)),
        'max_nonself_distance_at_5': float(np.max(non_self)),
    })

mcp_distance_df = pd.DataFrame(rows).sort_values('seed_doc_id')
mean_distance_overall = float(mcp_distance_df['mean_nonself_distance_at_5'].mean())

# 2) Agreement between MCP tools for DocId=2
tool_a_doc2 = mcp_find_top5[2]
tool_b_doc2 = mcp_execute_top5_doc2

set_overlap = len(set(tool_a_doc2) & set(tool_b_doc2))
set_union = len(set(tool_a_doc2) | set(tool_b_doc2))
jaccard_at_5 = set_overlap / set_union if set_union else np.nan
order_exact_match = tool_a_doc2 == tool_b_doc2

mcp_agreement_df = pd.DataFrame([
    {
        'seed_doc_id': 2,
        'jaccard_at_5_between_tools': jaccard_at_5,
        'order_exact_match': order_exact_match,
        'tool_a_top5_doc_ids': tool_a_doc2,
        'tool_b_top5_doc_ids': tool_b_doc2,
    }
])

print(f'MCP mean non-self distance@5 across seeds: {mean_distance_overall:.4f}')
display(mcp_distance_df)

print('MCP tool agreement check (find_similar_docs_by_doc_id vs execute_entity):')
display(mcp_agreement_df)

MCP mean non-self distance@5 across seeds: 0.2987


,seed_doc_id,mean_nonself_distance_at_5,max_nonself_distance_at_5
0,1,0.270325,0.342350
1,2,0.305822,0.339154
2,3,0.319949,0.373194


MCP tool agreement check (find_similar_docs_by_doc_id vs execute_entity):


,seed_doc_id,jaccard_at_5_between_tools,order_exact_match,tool_a_top5_doc_ids,tool_b_top5_doc_ids
0,2,1.0,True,"[2, 7, 18, 21, 3]","[2, 7, 18, 21, 3]"


: 

## 9) Model Card Style Note (Short)

### What this retrieval system is good for
- Finding semantically similar customer feedback across multiple languages when phrasing differs.
- Surfacing candidate evidence for recurring themes (for example: delivery delay, support quality, refund friction).
- Producing fast, reviewable top-k slices for analyst triage and theme discovery.
- Supporting mixed-source retrieval over reviews and support conversations.

### What this system is not good for
- It is not a final classifier or truth engine; similarity does not prove thematic correctness.
- It is not robust to subtle policy/legal distinctions without a downstream rule layer.
- It is not calibrated for fairness, harm, or compliance decisions.
- It is not a replacement for human annotation and adjudication.

### SQL MCP evidence included in this artifact
- Incorporated MCP tool 1: `find_similar_docs_by_doc_id`.
- Incorporated MCP tool 2: `execute_entity` on `FindSimilarDocsByDocId`.
- Added in-notebook MCP calculations:
  - Mean non-self distance@5 across seed docs.
  - Tool-agreement check (Jaccard@5 and exact order match) between both MCP paths.
- This MCP cross-check complements, but does not replace, hand-labeled Precision@5.

### Overclaim and how we reined it in
- Potential overclaim: "The system reliably identifies cross-language themes."
- Reined-in statement: "The system provides a useful cross-language retrieval baseline. Reliability claims require hand-labeled Precision@5, error analysis, and MCP consistency checks before production use."